In [ ]:
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

FONTES = [
    "santos_cet",
    "santos_sepref",
    "osasco_atendimento_cras",
    "osasco_atendimento_trabalhador"
]

DATE_COLS = ["data_criacao", "data_finalizacao", "data_carga"]


def union_bronze(sufixo: str, fontes: list) -> DataFrame:
    sdfs = [spark.table(f"bronze.{sufixo}_{f}") for f in fontes]
    return reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), sdfs)


def cast_date_cols(sdf: DataFrame, cols: list) -> DataFrame:
    for col_name in cols:
        if col_name in sdf.columns:
            sdf = sdf.withColumn(col_name, F.to_timestamp(F.col(col_name)))
    return sdf


silver_map = {
    "silver.fato_solicitacoes": union_bronze("fato_solicitacoes", FONTES),
    "silver.fato_campos":       union_bronze("fato_campos",       FONTES),
    "silver.fato_etapas":       union_bronze("fato_etapas",       FONTES),
}

for nome_tabela, sdf in silver_map.items():
    sdf = cast_date_cols(sdf, DATE_COLS)
    sdf.write.mode("overwrite").format("delta").saveAsTable(nome_tabela)
    print(f"✓ {nome_tabela}: {sdf.count()} linhas")